## SVD 3D assets

Generates 3D step images and transparent PNG frames for canvas playback.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from PIL import Image
from pathlib import Path

BASE_COLOR = "#00c2ff"
WIRE_COLOR = "#0b5ea8"
AXIS_COLORS = ["#e74c3c", "#27ae60", "#2980b9"]
MARKER_COLOR = "#ff3b30"
TITLE_COLOR = "#d6d6d6"

VIEW_ELEV = 22
VIEW_AZIM = 35

STATIC_SIZE = (4.2, 4.2)
STATIC_DPI = 200
FRAME_SIZE = (3.2, 3.2)
FRAME_DPI = 160

SUBPLOT = {
    "left": 0.04,
    "right": 0.96,
    "bottom": 0.04,
    "top": 0.90,
}

marker = np.array([0.85, 0.35, 0.2])


def rot_x(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1, 0, 0], [0, c, -s], [0, s, c]])


def rot_y(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])


def rot_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1]])


def normalize(v):
    nrm = np.linalg.norm(v)
    if nrm == 0:
        return v
    return v / nrm


def view_dir():
    az = np.deg2rad(VIEW_AZIM)
    el = np.deg2rad(VIEW_ELEV)
    return normalize(np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)]))


def sphere_grid(n=60):
    u = np.linspace(0, 2 * np.pi, n)
    v = np.linspace(0, np.pi, n)
    x = np.outer(np.cos(u), np.sin(v))
    y = np.outer(np.sin(u), np.sin(v))
    z = np.outer(np.ones_like(u), np.cos(v))
    return x, y, z


def apply(M, X, Y, Z):
    pts = np.stack([X.ravel(), Y.ravel(), Z.ravel()], axis=0)
    t = M @ pts
    return (t[0].reshape(X.shape), t[1].reshape(Y.shape), t[2].reshape(Z.shape))


def shaded_colors(X, Y, Z, base_color=BASE_COLOR, light_dir=(1.4, -0.6, 1.0)):
    Xu, Xv = np.gradient(X)
    Yu, Yv = np.gradient(Y)
    Zu, Zv = np.gradient(Z)
    nx = Yu * Zv - Zu * Yv
    ny = Zu * Xv - Xu * Zv
    nz = Xu * Yv - Yu * Xv
    norm = np.sqrt(nx * nx + ny * ny + nz * nz)
    norm = np.where(norm == 0, 1.0, norm)
    nx, ny, nz = nx / norm, ny / norm, nz / norm
    lx, ly, lz = np.array(light_dir, dtype=float)
    l_norm = np.sqrt(lx * lx + ly * ly + lz * lz)
    lx, ly, lz = lx / l_norm, ly / l_norm, lz / l_norm
    shade = np.clip(nx * lx + ny * ly + nz * lz, 0, 1)
    shade = np.power(shade, 0.55)
    r, g, b = mpl.colors.to_rgb(base_color)
    base = np.array([r, g, b])
    highlight = base + (1.0 - base) * 0.65
    shadow = base * 0.7
    return shadow[None, None, :] * (1 - shade[:, :, None]) + highlight[None, None, :] * shade[:, :, None]


def draw_axes(ax, M, vdir):
    axes = M @ np.eye(3)
    for i, color in enumerate(AXIS_COLORS):
        v = axes[:, i]
        if float(np.dot(v, vdir)) <= 0:
            continue
        ax.quiver(0, 0, 0, v[0], v[1], v[2], color=color, linewidth=2, arrow_length_ratio=0.08)


def draw_marker(ax, v):
    ax.plot([0, v[0]], [0, v[1]], [0, v[2]], color="#ffffff", linewidth=4, alpha=0.9)
    ax.plot([0, v[0]], [0, v[1]], [0, v[2]], color=MARKER_COLOR, linewidth=2.6)
    ax.scatter(
        [v[0]],
        [v[1]],
        [v[2]],
        color=MARKER_COLOR,
        s=60,
        edgecolors="#ffffff",
        linewidths=1.4,
        depthshade=False,
    )


def style_ax(ax, lim, title):
    ax.set_title(title, fontsize=11, pad=8, color=TITLE_COLOR)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_zlim(-lim, lim)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_zticks([])
    ax.set_facecolor((0, 0, 0, 0))
    ax.set_box_aspect((1, 1, 1))
    ax.set_proj_type("persp")
    ax.view_init(elev=VIEW_ELEV, azim=VIEW_AZIM)
    ax.grid(False)
    for axis in (ax.xaxis, ax.yaxis, ax.zaxis):
        axis.pane.set_facecolor((1, 1, 1, 0.0))
        axis.pane.set_edgecolor((1, 1, 1, 0.0))


def draw_scene(ax, M, title, X, Y, Z, lim):
    Xp, Yp, Zp = apply(M, X, Y, Z)
    colors = shaded_colors(Xp, Yp, Zp)
    ax.plot_surface(
        Xp,
        Yp,
        Zp,
        rstride=2,
        cstride=2,
        facecolors=colors,
        linewidth=0,
        antialiased=True,
        shade=False,
    )
    ax.plot_wireframe(
        Xp,
        Yp,
        Zp,
        rstride=7,
        cstride=7,
        color=WIRE_COLOR,
        alpha=0.14,
        linewidth=0.45,
    )
    vdir = view_dir()
    draw_axes(ax, M, vdir)
    draw_marker(ax, normalize(M @ marker))
    style_ax(ax, lim, title)


def plot_step(title, M, out_path, X, Y, Z, lim):
    fig = plt.figure(figsize=STATIC_SIZE, dpi=STATIC_DPI)
    ax = fig.add_subplot(1, 1, 1, projection="3d")
    fig.patch.set_alpha(0)
    fig.patch.set_facecolor((0, 0, 0, 0))
    draw_scene(ax, M, title, X, Y, Z, lim)
    fig.subplots_adjust(**SUBPLOT)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(out_path, transparent=True)
    plt.close(fig)


def render_frame(M, title, X, Y, Z, lim, size=FRAME_SIZE, dpi=FRAME_DPI):
    fig = plt.figure(figsize=size, dpi=dpi)
    ax = fig.add_subplot(1, 1, 1, projection="3d")
    fig.patch.set_alpha(0)
    fig.patch.set_facecolor((0, 0, 0, 0))
    draw_scene(ax, M, title, X, Y, Z, lim)
    fig.subplots_adjust(**SUBPLOT)
    fig.canvas.draw()
    frame = np.asarray(fig.canvas.renderer.buffer_rgba()).copy()
    plt.close(fig)
    return frame


def save_rgba_png(frame_rgba, path):
    Image.fromarray(frame_rgba).save(path)


In [ ]:
# Build a clean SVD-style 3D transform
angle_v_z = np.deg2rad(25)
angle_v_y = np.deg2rad(20)
angle_u_x = np.deg2rad(-20)
angle_u_z = np.deg2rad(-15)

Vt = rot_z(angle_v_z) @ rot_y(angle_v_y)
U = rot_x(angle_u_x) @ rot_z(angle_u_z)
Sigma = np.diag([1.5, 1.0, 0.6])

X, Y, Z = sphere_grid()
lim = float(Sigma.max() * 1.35)

steps = [
    ("Unit sphere", np.eye(3), "svd-3d-1-unit.png"),
    ("Rotate $V^T$", Vt, "svd-3d-2-rotate-vt.png"),
    (r"Scale $\Sigma$", Sigma @ Vt, "svd-3d-3-scale-sigma.png"),
    ("Rotate $U$ (A)", U @ Sigma @ Vt, "svd-3d-4-rotate-u.png"),
]

out_dir = Path("docs/public/images/math/svd")

for title, M, name in steps:
    plot_step(title, M, out_dir / name, X, Y, Z, lim)


frames_dir = out_dir / "frames"
if frames_dir.exists():
    for item in frames_dir.glob("svd-3d-frame-*.png"):
        item.unlink()
else:
    frames_dir.mkdir(parents=True, exist_ok=True)

frame_count = 60
seg = frame_count // 3
last_seg = frame_count - seg * 2

idx = 0
for i in range(seg):
    t = i / (seg - 1) if seg > 1 else 1.0
    M = rot_z(angle_v_z * t) @ rot_y(angle_v_y * t)
    frame = render_frame(M, "Rotate $V^T$", X, Y, Z, lim)
    save_rgba_png(frame, frames_dir / f"svd-3d-frame-{idx:03d}.png")
    idx += 1

for i in range(seg):
    t = i / (seg - 1) if seg > 1 else 1.0
    scale = np.diag([
        1.0 + t * (Sigma[0, 0] - 1.0),
        1.0 + t * (Sigma[1, 1] - 1.0),
        1.0 + t * (Sigma[2, 2] - 1.0),
    ])
    M = scale @ Vt
    frame = render_frame(M, r"Scale $\Sigma$", X, Y, Z, lim)
    save_rgba_png(frame, frames_dir / f"svd-3d-frame-{idx:03d}.png")
    idx += 1

for i in range(last_seg):
    t = i / (last_seg - 1) if last_seg > 1 else 1.0
    M = (rot_x(angle_u_x * t) @ rot_z(angle_u_z * t)) @ Sigma @ Vt
    frame = render_frame(M, "Rotate $U$ (A)", X, Y, Z, lim)
    save_rgba_png(frame, frames_dir / f"svd-3d-frame-{idx:03d}.png")
    idx += 1
